# 03. QKMS key lifecycle와 fail-closed 운영

목표: QKD가 만든 key material을 application에 안전하게 전달할 때 필요한 key ID, peer binding, TTL, one-time consume와 exhaustion policy를 구현한다. 실제 encryption이나 ETSI GS QKD 014 server 구현이 아니다.

## 1. Key record

Key byte를 log나 exception에 노출하지 않는다. 여기서는 상태 전이를 보기 위해 metadata만 출력한다.

In [ ]:
from dataclasses import dataclass
from enum import Enum


class KeyState(str, Enum):
    AVAILABLE = "available"
    CONSUMED = "consumed"
    EXPIRED = "expired"


@dataclass
class KeyRecord:
    key_id: str
    peer: str
    material: bytes
    created_at: int
    expires_at: int
    state: KeyState = KeyState.AVAILABLE


class KeyUnavailable(RuntimeError):
    pass


class ToyKeyManager:
    def __init__(self) -> None:
        self._records: dict[str, KeyRecord] = {}
        self.audit: list[dict] = []

    def add(self, record: KeyRecord) -> None:
        if record.key_id in self._records:
            raise ValueError("duplicate key_id")
        self._records[record.key_id] = record
        self.audit.append({"event": "added", "key_id": record.key_id, "peer": record.peer})

    def consume(self, key_id: str, peer: str, now: int) -> bytes:
        record = self._records.get(key_id)
        if record is None or record.peer != peer:
            raise KeyUnavailable("unknown key or peer mismatch")
        if now >= record.expires_at:
            record.state = KeyState.EXPIRED
            self.audit.append({"event": "expired", "key_id": key_id})
            raise KeyUnavailable("key expired")
        if record.state is not KeyState.AVAILABLE:
            raise KeyUnavailable("key already consumed or expired")
        record.state = KeyState.CONSUMED
        self.audit.append({"event": "consumed", "key_id": key_id, "peer": peer})
        return record.material

## 2. 정상 소비와 재사용 차단

동일 key ID를 두 번 전달하지 않는 fail-closed 동작을 확인한다. Production에서는 transaction과 crash recovery가 추가로 필요하다.

In [ ]:
manager = ToyKeyManager()
manager.add(KeyRecord("qkd-0001", "site-b", b"demo-material-not-for-production", 100, 200))
material = manager.consume("qkd-0001", "site-b", now=150)
assert material.startswith(b"demo")

try:
    manager.consume("qkd-0001", "site-b", now=151)
    raise AssertionError("reuse should have failed")
except KeyUnavailable as error:
    print("reuse blocked:", error)

print("audit metadata:", manager.audit)
assert all("material" not in event for event in manager.audit)

## 3. Expiration과 peer binding

다른 peer가 key를 요청하거나 TTL이 지난 경우 key를 반환하지 않는다.

In [ ]:
manager.add(KeyRecord("qkd-0002", "site-b", b"second-demo-material", 100, 120))

for peer, now in (("site-c", 110), ("site-b", 130)):
    try:
        manager.consume("qkd-0002", peer, now)
        raise AssertionError("invalid request should have failed")
    except KeyUnavailable as error:
        print(f"blocked peer={peer} now={now}: {error}")

assert manager._records["qkd-0002"].state is KeyState.EXPIRED

## 4. Production 확장 과제

- Database transaction으로 `available → reserved → consumed`를 원자화한다.
- Alice/Bob KME의 key ID 불일치와 retry를 idempotent하게 처리한다.
- Key material은 HSM boundary 밖에서 평문으로 저장하지 않는다.
- Inventory low-watermark, QBER, generation rate와 expiry를 monitoring한다.
- Key 부족 시 plaintext나 장기 재사용으로 조용히 downgrade하지 않는다.
- 승인된 PQC/classical fallback이 있다면 algorithm과 이유를 audit한다.
- Concurrent consume, process crash, clock rollback과 network partition test를 추가한다.